In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Convolutional Network Design Spaces

Most networks in this chapter were designed by hand. Each design fixes a few
numbers for every stage: how many blocks it repeats, how many channels they
produce, and what the block looks like. Where do these numbers come from, and
are there rules for choosing them? The networks we built give little guidance.
ResNet-18 repeats its blocks $(2, 2, 2, 2)$ times across four stages, ResNet-50
$(3, 4, 6, 3)$ times, and ConvNeXt-T $(3, 3, 9, 3)$ times. ResNet's depths come
with no stated rule, and ConvNeXt-T's were copied from Swin and checked once.
The choices interact: at a fixed budget of computation, a deeper stage must be
narrower, and a block with grouped convolutions can afford more channels than
one without. Could we not simply try every combination? Each stage of the
networks below is described by four numbers, and with four stages and only two
values for each number there are already $2^{16} = 65{,}536$ variants, each
needing its own training run. Worse, a single good network, however it was
found, does not say which of its choices mattered, so nothing carries over to a
new compute budget or a new block.

Neural architecture search, which found EfficientNet's base network
(that section), automates the hand design
[@zoph2016neural; @liu2018darts]: it searches a set of candidate networks
for the one with the best estimated accuracy. Like hand design, however, it
returns one network rather than a reason, and compound scaling only enlarges
that network. @Radosavovic.Kosaraju.Girshick.ea.2020 take a different
route and compare whole *design spaces*: populations of networks defined by a
template and a few constraints. They sample networks from a space, train each
briefly, and keep a constraint if it improves the error distribution of the
whole population. The constraints that survive are design rules, and the
networks that satisfy all of them form the RegNet family. We build the
template, read the paper's evidence from ImageNet, where each comparison trains
500 networks, and then compare two small populations of our own on
Fashion-MNIST that differ in the rule the paper found to matter most. Finally,
we train a small RegNet.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## The AnyNet Design Space

The networks of this chapter share one template, the stem, body, and head of
that section. The stem processes the raw image with one or a few
convolutions and usually lowers its resolution. The body does most of the work.
It is a sequence of *stages*: each stage operates at one resolution and repeats
one kind of *block*, and each new stage halves the resolution and widens the
channels. The head, global average pooling and a linear layer, turns the last
feature map into class scores. ResNet's body has four stages, which end at
$1/4$, $1/8$, $1/16$, and $1/32$ of the input resolution. One reason why this
layout has persisted from ResNet to ConvNeXt is that detection and segmentation
networks read features from all four stages, coarse ones for large objects and
fine ones for small objects (that section).

### Stage Parameters of Hand-Designed Networks

the table lists the stages of four networks from
this chapter. Their widths double from stage to stage in every row. Their
depths follow no visible rule.

<!-- tbl-caption:Stage depths and widths of four hand-designed networks (ImageNet versions). -->

| Network | Blocks per stage | Output channels per stage | Block |
|:--|:--|:--|:--|
| ResNet-18 (that section) | 2, 2, 2, 2 | 64, 128, 256, 512 | two $3 \times 3$ convolutions |
| ResNet-50 [@He.Zhang.Ren.ea.2016] | 3, 4, 6, 3 | 256, 512, 1024, 2048 | $1 \times 1$, $3 \times 3$, $1 \times 1$ bottleneck |
| ResNeXt-50 (that section) | 3, 4, 6, 3 | 256, 512, 1024, 2048 | bottleneck with 32 groups |
| ConvNeXt-T (that section) | 3, 3, 9, 3 | 96, 192, 384, 768 | $7 \times 7$ depthwise, inverted bottleneck |

The *AnyNet* design space of @Radosavovic.Kosaraju.Girshick.ea.2020
fixes the template and leaves these numbers free
(the figure). The stem is one $3 \times 3$ convolution with
stride 2 and $c_0 = 32$ output channels, followed by batch normalization and
a ReLU. In the paper it reads RGB images; ours reads the single channel of
Fashion-MNIST. The body has four stages, and stage $i$ repeats one block
$d_i$ times. The block is the ResNeXt block of
the figure: a $1 \times 1$ convolution narrows the $c_i$
channels of the stage to $c_i / b_i$, a $3 \times 3$ convolution processes
them in groups of $g_i$ channels each, and a second $1 \times 1$ convolution
restores $c_i$ channels. The symbol $b_i$ is the paper's, for the
*bottleneck ratio*; the number of bottleneck channels, which
that section called $b$, is $c_i / b_i$ here. Batch
normalization follows each convolution, and a
residual connection adds the input. The first block of each stage halves the
resolution with stride 2 and passes its shortcut through a $1 \times 1$
convolution so that the shapes match. The head is global average pooling and
one fully connected layer.

![The AnyNet design space: a stem, a body of four stages, and a head. Stage $\mathit{i}$ repeats a ResNeXt block $\mathit{d_i}$ times; the first block of each stage halves the resolution. The annotations give the number of channels and the resolution after each part for an $\mathit{r} \times \mathit{r}$ input; the head outputs $\mathit{n}$ class scores. The free choices per stage are the depth $\mathit{d_i}$, the width $\mathit{c_i}$, the group width $\mathit{g_i}$, and the bottleneck ratio $\mathit{b_i}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-anynet.svg)

Each stage has four free numbers: its depth $d_i$, its width $c_i$, the
bottleneck ratio $b_i \geq 1$, and the group width $g_i$, the number of
channels per group. A warning about $g_i$ is in order: the paper counts
channels per group, whereas $g$ in that section counted groups, and
the `groups` argument of `d2l.ResNeXtBlock` keeps the latter meaning. Four
stages give 16 numbers. The paper samples depths up to 16, widths up to 1024 in
multiples of 8, bottleneck ratios from $\{1, 2, 4\}$, and group widths from the
powers of two up to 32, which makes about
$(16 \cdot 128 \cdot 3 \cdot 6)^4 \approx 2 \times 10^{18}$ networks:
far too many to enumerate, but easy to sample from. For $224 \times 224$
ImageNet images the stem and the four stages each halve the resolution, so
the body ends at $224 / 2^5 = 7$ pixels on a side.

### Implementing AnyNet

To work with a design space, we need code that builds any of its members from
the stage parameters. We start with the stem, a three-layer sequential block.

In [ ]:
class AnyNet(d2l.Classifier):
    def stem(self, num_channels):
        return nn.Sequential(
            nn.LazyConv2d(num_channels, kernel_size=3, stride=2, padding=1,
                          bias=False),
            nn.LazyBatchNorm2d(), nn.ReLU())

A stage repeats `d2l.ResNeXtBlock` from that section. Its
arguments are the width $c_i$, the number of groups, and `bot_mul`
$= 1/b_i$; the group width is the bottleneck width $c_i/b_i$ divided by the
number of groups. Only the first block of the stage has stride 2.

In [ ]:
@d2l.add_to_class(AnyNet)
def stage(self, depth, num_channels, groups, bot_mul):
    blk = []
    for i in range(depth):
        if i == 0:
            blk.append(d2l.ResNeXtBlock(num_channels, groups, bot_mul,
                use_1x1conv=True, strides=2))
        else:
            blk.append(d2l.ResNeXtBlock(num_channels, groups, bot_mul))
    return nn.Sequential(*blk)

The network is the stem, one stage per entry of the tuple `arch`, and the head.
Each entry of `arch` holds the four numbers
`(depth, num_channels, groups, bot_mul)` of one stage. As in
that section, every convolution and linear layer starts from He
initialization (that section), and the convolutions have no bias,
since batch normalization follows each of them.

In [ ]:
@d2l.add_to_class(AnyNet)
def __init__(self, arch, stem_channels, lr=0.1, num_classes=10):
    super(AnyNet, self).__init__()
    self.save_hyperparameters()
    self.net = nn.Sequential(self.stem(stem_channels))
    for i, s in enumerate(arch):
        self.net.add_module(f'stage{i+1}', self.stage(*s))
    self.net.add_module('head', nn.Sequential(
        nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        nn.LazyLinear(num_classes)))

@d2l.add_to_class(AnyNet)
def weight_init(self, module):
    # The Trainer calls this after the first minibatch fixed the shapes
    if type(module) in (nn.Linear, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
        if module.bias is not None:
            nn.init.zeros_(module.bias)

Every network of the space is trained with one recipe, so we attach the
optimizer to `AnyNet` itself. It is the one under which
@Radosavovic.Kosaraju.Girshick.ea.2020 found their rules, and that of
recipe A in that section: SGD with momentum 0.9, whose steps
follow a running sum of recent gradients (that section;
that section treats it fully), and a weight decay that is added
to the gradient (that section). The learning rate follows the
schedule of this chapter (`d2l.cosine_warmup` of that section):
linear warmup over the first tenth of the `num_steps` updates to the peak rate
`lr`, then a cosine decay to zero. The paper uses a decay of
$5 \times 10^{-5}$ for 100 epochs of ImageNet. For our short runs on
Fashion-MNIST we use the larger $10^{-3}$, the best for the RegNetX32 of
that section among the handful of values we tried and as good as
any for ResNet-18.

In [ ]:
@d2l.add_to_class(AnyNet)
def configure_optimizers(self):
    optim = torch.optim.SGD(self.parameters(), lr=self.lr, momentum=0.9,
                            weight_decay=1e-3)  # Decay added to the gradient
    warmup = self.num_steps // 10
    schedule = torch.optim.lr_scheduler.LambdaLR(
        optim, lambda t: d2l.cosine_warmup(t, self.num_steps, warmup))
    return optim, schedule  # The Trainer steps it after every minibatch

## Comparing Design Spaces

Two design spaces can be compared without finding the best network in
either. @radosavovic2019network make four assumptions that let
a population of briefly trained networks stand in for the space:

1. Good design principles exist, so that many networks satisfying them
   perform well. A distribution over networks is then a sensible object to
   optimize: there are many good needles in the haystack.
1. Accuracy after a few passes through the data ranks networks nearly as
   accuracy after full training does. This is why the paper trains each
   network for only 10 epochs, and we train ours below for one.
1. Results for small networks carry over to larger ones, so the search can
   run at a low compute budget and be checked at a high one at the end.
1. The effects of different design choices are roughly independent, so each
   can be studied separately.

None of the four is obvious, and the second and third are rather bold; the
exercises test them at small scale.

Granting them, we sample $n$ networks from a space, train each, and record its
error. The *empirical cumulative distribution function* (CDF) of the errors
$e_1, \ldots, e_n$ gives, for every threshold $e$, the fraction of sampled
networks whose error is at most $e$:

$$\hat{F}(e) = \frac{1}{n}\sum_{j=1}^n \mathbf{1}(e_j \leq e).$$

A curve that rises early belongs to a space in which many networks reach low
error. If the curve of one space lies above that of another at every $e$, a
random network from the first space is at least as likely as one from the
second to beat any threshold. This compares populations, not best networks:
the best network of the two spaces may be the same.

@Radosavovic.Kosaraju.Girshick.ea.2020 sample 500 networks from each
space at a budget of about 400 million multiply-adds (the paper's 400 MFLOPs),
train each for 10 epochs on ImageNet, and compare the CDFs in
the figure. Each step adds one constraint to the unconstrained
AnyNetX$_A$ space. Using one bottleneck ratio $b$ for all stages (AnyNetX$_B$)
and then also one group width $g$ (AnyNetX$_C$) leaves the curve where it was,
with a mean error near 49%. This is a gain in itself: the two constraints
remove six of the 16 numbers at no measurable cost, and the space becomes
smaller and easier to search. The constraint that matters is on the widths.
Requiring them to increase across stages, $c_{i+1} \geq c_i$, lowers the mean
error of the sampled networks by six points, to about 43%, while equal or
decreasing widths raise it to 56% and 63%. Requiring depths to increase as
well, $d_{i+1} \geq d_i$, gains another half point; constant or decreasing
depths lose ground.

![Error CDFs of design spaces, reproduced from @Radosavovic.Kosaraju.Girshick.ea.2020: 500 networks sampled from each space, each trained for 10 epochs on ImageNet at about 400 million multiply-adds. Legends give the minimum and mean top-1 error in percent; $w_i$ is the paper's symbol for the width $c_i$. Top: one bottleneck ratio for all stages (AnyNetX$_B$) against none (AnyNetX$_A$), then one group width as well (AnyNetX$_C$). Bottom: widths that increase, stay equal, or decrease across stages; then depths that do the same in the space with increasing widths (AnyNetX$_D$).](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/regnet-fig.png)

How far does this evidence reach? It is specific to the paper's block,
budget, data, and training protocol, but the method is not, and we can apply
it at our own scale to the rule that mattered most. Our two design spaces use
two stages, a stem of 32 channels, and ResNeXt blocks with $b = 1$ and a group
width of 16 channels. Stage $i$ draws its depth $d_i$ from 1 to 8 and its
width $c_i$ from the multiples of 16 up to 256. The spaces differ in one rule
only: $c_1 < c_2$ in one, $c_1 > c_2$ in the other. As in the paper, the
networks are matched in compute. We count the multiply-adds of a network as
for AlexNet in that section, adding up the convolutions of the stem,
of every block, and of the shortcuts.

In [ ]:
def mult_adds(arch, r, stem_channels=32, group_width=16, num_classes=10):
    """Multiply-adds of an AnyNet with k = 1 for one r x r image; arch holds
    (depth, width) for each stage."""
    r //= 2  # The stem halves the resolution
    total, c_in = r * r * 9 * stem_channels, stem_channels  # Grayscale input
    for d, c in arch:
        total += r * r * c_in * c  # First 1 x 1 convolution, before the stride
        r //= 2
        total += r * r * c_in * c  # 1 x 1 convolution of the shortcut
        total += r * r * c * c * (2 * d - 1)  # The other 1 x 1 convolutions
        total += r * r * 9 * group_width * c * d  # Grouped 3 x 3 convolutions
        c_in = c
    return total + c_in * num_classes  # Linear head

We sample a network by drawing two depths and two different widths, ordering
the widths as the space requires, and rejecting the draw unless its
multiply-adds lie within 20% of a budget. The budget is 112 million per
$96 \times 96$ image, that of the RegNetX32 we train in
that section.

In [ ]:
def sample_arch(rng, increasing, r=96, budget=112e6):
    """Depths and widths of two stages at about `budget` multiply-adds."""
    while True:
        depths = rng.integers(1, 9, size=2).tolist()
        widths = sorted(rng.choice(np.arange(16, 257, 16), 2, replace=False)
                        .tolist(), reverse=not increasing)
        arch = tuple(zip(depths, widths))
        if abs(mult_adds(arch, r) / budget - 1) <= 0.2:
            return arch

rng = np.random.default_rng()
for increasing in (True, False):
    arch = sample_arch(rng, increasing)
    print(arch, f'{mult_adds(arch, 96) / 1e6:.0f} million multiply-adds')

At equal multiply-adds, a network with decreasing widths is wide where the
maps are large, and narrow where they are small. Each sampled network trains
for one epoch, a tenth of the training runs of that section, with
the optimizer of `AnyNet`, and we record its validation error.

In [ ]:
def population_errors(rng, increasing, data, n, num_epochs=1):
    """Validation errors of n networks sampled from one design space."""
    errors = []
    for _ in range(n):
        arch = sample_arch(rng, increasing)
        model = AnyNet(tuple((d, c, c // 16, 1) for d, c in arch), 32)
        model.num_steps = num_epochs * len(data.train_dataloader())
        trainer = d2l.Trainer(max_epochs=num_epochs, num_gpus=1)
        trainer.board.display = False  # We plot the errors ourselves
        trainer.fit(model, data)
        errors.append(1 - float(trainer.history['val_acc'][-1]))
    return np.array(errors)

We sample 12 networks from each space, 24 training runs of one epoch each.

In [ ]:
data = d2l.FashionMNIST(batch_size=128, resize=(96, 96))
spaces = {'increasing widths': True, 'decreasing widths': False}
errors = {name: population_errors(rng, increasing, data, 12)
          for name, increasing in spaces.items()}

The empirical CDF the equation of each population is a step function that
rises by $1/n$ at each sorted error.

In [ ]:
def ecdf(e):
    """Corners of the step function of the empirical CDF of e."""
    e = np.sort(e)
    return np.repeat(e, 2), np.repeat(np.arange(len(e) + 1) / len(e), 2)[1:-1]

X, Y = zip(*(ecdf(e) for e in errors.values()))
d2l.plot(list(X), list(Y), xlabel='validation error',
         ylabel='fraction of networks', legend=list(errors),
         legend_loc='outside', figsize=(5, 2.5))
for name, e in errors.items():
    print(f'{name}: mean error {e.mean():.3f}, minimum {e.min():.3f}')

The two curves interleave. Over repeated runs, the mean errors of the two
spaces differ by less than a point, near 12%, usually in favor of increasing
widths, and the best networks of both spaces reach about 11%. A rule that is
worth twenty points of mean error on ImageNet is worth less than one here,
and twelve samples per space cannot resolve a difference that small
reliably. This is a rather sobering outcome for our first population, but
not a surprising one. After one epoch on an easy dataset most networks of
either space do well, which leaves the rule little to separate. Which design
rules matter, and how much, depends on the data, and the rules of
@Radosavovic.Kosaraju.Girshick.ea.2020 were found on ImageNet.

## RegNet

### Design Principles

The constraints that survived define the AnyNetX$_E$ space:

* one bottleneck ratio $b$ for all stages;
* one group width $g$ for all stages;
* widths that do not decrease across stages, $c_{i} \leq c_{i+1}$;
* depths that do not decrease across stages, $d_{i} \leq d_{i+1}$.

The best networks of AnyNetX$_E$ share one more regularity. If we number the
blocks $j = 0, 1, \ldots$ through the whole body, the width of block $j$
grows roughly linearly,

$$u_j = u_0 + a j,$$

with an initial width $u_0$ and a slope $a > 0$. Since all blocks of a stage
must share one width, RegNet snaps each $u_j$ to a nearby value on a geometric
grid, whose successive values differ by a constant factor, as the widths of the
hand-designed networks in the table do. Blocks that
land on the same value form a stage, which turns the line into a staircase
(problem 2 works one out). The paper
reports a further finding in the resulting space, and it runs against the
practice of ResNet-50 and ResNeXt-50: the best networks use $b = 1$, so the
block does not narrow its channels at all. The *RegNetX* family consists of the
networks with quantized linear widths and $b = 1$. It is described by a handful
of numbers per compute budget instead of 16.

### RegNetX32 on Fashion-MNIST

We now build a small network that follows the rules and compare it with
ResNet-18. Fashion-MNIST images, upsampled to $96 \times 96$ as in
that section, carry only $28 \times 28$ pixels of detail, and the stem
and one stage already bring them to $24 \times 24$. Four stages, as in the
paper's ImageNet models, would put the last two on $6 \times 6$ and
$3 \times 3$ maps. We therefore use two, which keep every block at
$24 \times 24$ or $12 \times 12$ (problem
1 tries four). The stem has 32 channels;
stage 1 has $d_1 = 4$ blocks of width $c_1 = 64$ and stage 2 has $d_2 = 6$
blocks of width $c_2 = 160$. We picked these numbers by hand so
that widths and depths increase; nothing was sampled or searched. The
bottleneck ratio is $b = 1$, and both stages share the group width $g = 16$, so
their $3 \times 3$ convolutions have $64/16 = 4$ and $160/16 = 10$ groups.
Counting one stem convolution, three convolutions per block, and the linear
head gives 32 layers, hence the name.

In [ ]:
class RegNetX32(AnyNet):
    def __init__(self, lr=0.1, num_classes=10, num_steps=None):
        stem_channels, group_width, bot_mul = 32, 16, 1
        depths, channels = (4, 6), (64, 160)
        super().__init__(  # The number of groups is c_i / g
            tuple((d, c, c // group_width, bot_mul)
                  for d, c in zip(depths, channels)),
            stem_channels, lr, num_classes)
        self.num_steps = num_steps  # Length of the learning-rate schedule

The layer summary shows the resolution falling from 96 to 48 in the stem and
then to 24 and 12 in the two stages, while the channels widen.

In [ ]:
RegNetX32().layer_summary((1, 1, 96, 96))

Counted as for AlexNet in that section, RegNetX32 performs 112 million
multiply-adds per $96 \times 96$ image. ResNet-18 performs 319 million, almost
three times as many.

How well does a network built from these rules do against ResNet-18? We train
on Fashion-MNIST at $96 \times 96$ with minibatches of 128 for 10 epochs, with
the optimizer of `AnyNet` at a peak rate of 0.1, so that the warmup takes the
first epoch.

In [ ]:
data = d2l.FashionMNIST(batch_size=128, resize=(96, 96))
model = RegNetX32(lr=0.1, num_steps=10 * len(data.train_dataloader()))
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
trainer.fit(model, data)

We count the parameters of the trained network and read off its final
validation accuracy.

In [ ]:
print(f'{d2l.num_params(model):,} parameters, validation accuracy '
      f'{trainer.history["val_acc"][-1]:.3f}')

For a fair comparison, ResNet-18 must be trained with the same optimizer.
Conveniently, it fits the AnyNet template. We override `stem` and `stage` with
the stem and the residual block `d2l.Residual` of that section, which
gives the network of that section, and it inherits the optimizer that we
attached to `AnyNet`. Each entry of `arch` now holds a depth, a width, and the
stride of the stage's first block.

In [ ]:
class ResNet18(AnyNet):
    def __init__(self, lr=0.1, num_classes=10, num_steps=None):
        super().__init__(((2, 64, 1), (2, 128, 2), (2, 256, 2), (2, 512, 2)),
                         64, lr, num_classes)
        self.num_steps = num_steps

    def stem(self, num_channels):
        return nn.Sequential(
            nn.LazyConv2d(num_channels, kernel_size=7, stride=2, padding=3,
                          bias=False),
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

    def stage(self, depth, num_channels, stride):
        blk = [d2l.Residual(num_channels, use_1x1conv=stride > 1,
                            strides=stride)]
        blk += [d2l.Residual(num_channels) for _ in range(depth - 1)]
        return nn.Sequential(*blk)

We train it on the same data for the same 10 epochs.

In [ ]:
model = ResNet18(lr=0.1, num_steps=10 * len(data.train_dataloader()))
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
trainer.fit(model, data)

In [ ]:
print(f'{d2l.num_params(model):,} parameters, validation accuracy '
      f'{trainer.history["val_acc"][-1]:.3f}')

ResNet-18 has 11,175,370 parameters and RegNetX32 519,658, less than 5% as
many, and both reach about 94% validation accuracy after 10 epochs. Over
three or more runs in each framework, the accuracies of the two networks
spread over the same range, from about 93.7% to 94.2%, so the difference
between the two printed accuracies is run-to-run noise. What does the tie
show? It cannot show that the rules produce better networks. The two
populations above tested one rule with everything else drawn alike; these two
networks differ in depth, width, and block all at once, and 94% will turn out
to be the ceiling of this dataset. It does show that ResNet-18 is far
larger than the task requires. A network that obeys the rules reaches the
same accuracy with about a third of the multiply-adds.

The optimizer is part of the result. With the plain SGD of
that section, the same RegNetX32 ends near 91%, while ResNet-18
reaches 93%. The rules were found and tested with momentum and weight decay,
and that is the setting in which RegNetX32 matches ResNet-18; the exercises
ask which of the two ingredients accounts for the gap.

Larger members of the design space gain nothing on a dataset this easy. The
paper's four-stage RegNetX-400MF, -800MF, and -1.6GF have 4.8 to 8.3 million
parameters. Trained with the same recipe, and with the last
batch-normalization scale of every block initialized to zero so that each
block starts as the identity (that section), without which
some runs of these deeper networks start badly, they end between 93% and
94%. At this
scale the dataset, not the architecture, limits the accuracy to about 94%.

Fewer multiply-adds do not make RegNetX32 faster to train, either: on a GPU,
an epoch takes about one and a half times as long as one of ResNet-18.
Multiply-adds count arithmetic, not time (that section), and
GPU libraries run grouped convolutions, like depthwise ones, well below the
arithmetic rate of dense ones. The paper's claim of speed is accordingly a
narrow one. @Radosavovic.Kosaraju.Girshick.ea.2020 report that
RegNets match or beat EfficientNets of equal multiply-adds in accuracy and
run up to five times faster on GPUs; against ResNets they find similar GPU
times only at equal numbers of activations, the values that all convolutions
output for one image.

## Summary

Hand-designed networks fix their stage depths and widths by intuition, and a
single good network does not reveal which of its choices mattered. A design
space is a population of networks defined by a template, here AnyNet: a
stem, four stages of ResNeXt blocks with a free depth, width, group width,
and bottleneck ratio each, and a head. Comparing the error distributions of
briefly trained samples shows, on ImageNet, that one bottleneck ratio and one
group width for all stages cost nothing, while widths and depths that grow
across stages help. On Fashion-MNIST, two populations of a dozen networks
each barely separate: there the width rule is worth less than a point of
mean error instead of twenty. RegNet adds a bottleneck ratio of 1 and widths
that grow linearly with the block index, which turns 16 free numbers into a
few. On Fashion-MNIST, a 32-layer RegNetX matches the accuracy of ResNet-18, about 94%, with less
than 5% of its parameters and a third of its multiply-adds, when both are
trained with momentum and weight decay. Whether the rules hold for other
blocks, such as those of ConvNeXt or of the vision transformers of
that section, is a question the same method can answer
(problem 8).

This section ends the chapter, and its experiment says something about the
others. On upsampled Fashion-MNIST, ResNet-18 already sits at about 94%,
which leaves the designs that followed it little room to show an advantage in
accuracy; for that we had to quote ImageNet results. What the chapter does
leave us with is a short list of parts that survived every redesign (blocks
repeated in stages, normalization, residual connections) and the habit of
asking whether a gain belongs to the network or to the way it was trained.

## Exercises

1. [code] **Four stages at matched budget.** `RegNetX32` uses two stages.
   Extend it to the four stages of the figure, keeping widths
   and depths nondecreasing across stages and the parameter count within 10%
   of `RegNetX32`. Compare validation accuracy and time per epoch on
   Fashion-MNIST at $96 \times 96$. Work out the feature-map size at which
   each stage operates and explain what a $3 \times 3$ convolution computes
   on the last one.

1. **Quantized linear widths.** @Radosavovic.Kosaraju.Girshick.ea.2020
   turn the observation that width should grow linearly with the block index
   into a rule with three numbers. For $d$ blocks, the target width of block
   $j$ is the linear rule of that section,
   $$u_j = u_0 + a\, j, \qquad j = 0, \ldots, d-1.$$
   Each block is snapped to a geometric grid with ratio $m$,
   $$s_j = \operatorname{round}\left(\frac{\log(u_j / u_0)}{\log m}\right), \qquad \tilde{u}_j = u_0\, m^{s_j}.$$
   Blocks with equal $\tilde{u}_j$ form one stage.
    1. For $d = 13$, $u_0 = 24$, $a = 36.44$, and $m = 2.49$ (the paper's
       $w_0$, $w_a$, and $w_m$), compute the number of stages, the depth of
       each, and the stage widths rounded to the nearest multiple of 8.
       Compare with the RegNetX-200MF configuration in the paper.
    1. Describe how the number of stages, and how closely $\tilde{u}_j$
       tracks $u_j$, change as $m \to 1$ and as $m$ grows large.
    1. A shared group width must divide every stage width. Which of your
       widths violate this for a group width of 16, and how would you adjust
       them?

1. [code] **RegNetY.** RegNetY adds the squeeze-and-excitation gate `SE`
   of that section to every block. Insert `SE` into
   `d2l.ResNeXtBlock` after its grouped
   $3 \times 3$ convolution, as @Radosavovic.Kosaraju.Girshick.ea.2020
   do, and build a RegNetY32 with the configuration of `RegNetX32`.
    1. Predict the added parameter count from the estimate of about
       $2c^2/r$ per gate and verify it against the model.
    1. Compare validation accuracy and time per epoch with `RegNetX32`.
       Where does the extra time come from, given that the gate adds almost
       no multiply-adds?

1. **Design spaces versus architecture search.** Neural architecture search
   [@zoph2016neural; @liu2018darts] selects one network from a search
   space; the method of that section
   constrains a distribution over networks.
    1. Explain why comparing empirical CDFs the equation cannot identify
       the best network in a space, and why that is not its purpose.
    1. Of the four assumptions listed in
       that section, which does a NAS method
       that trains candidates briefly also rely on, and which are specific
       to the design-space approach?
    1. Name one failure mode of each approach: what can go wrong when a
       single architecture is selected by a proxy score, and what can go
       wrong when a design space is constrained by rules fitted to a sample?

1. [code] **Breaking the rules.** ● The two populations of this section
   tested the rule on widths. Implement variants of `RegNetX32` that each
   violate one of the other design principles of AnyNetX$_E$ in
   that section, at roughly matched parameter count: depths
   that decrease across stages, group widths that differ between stages,
   and bottleneck ratios that differ between stages.
    1. Report validation accuracy for each violation against `RegNetX32`.
       Train every network three times: runs of one network differ by a
       few tenths of a point.
    1. Which of depth $d_i$, group width $g_i$, and bottleneck ratio $b_i$
       matters most here?
    1. Compare your ranking with the figure. Which violations
       did the sampled design spaces of
       @Radosavovic.Kosaraju.Girshick.ea.2020 also find harmless,
       and why does one network per violation give weaker evidence than an
       error CDF?

1. [code] **The optimizer.** With plain SGD at a peak rate of 0.1,
   `RegNetX32` ends several points below the accuracy it reaches with
   momentum and weight decay.
    1. Train it with momentum 0.9 and no weight decay, and with plain SGD
       at peaks of 0.1 and 1.0. Which change accounts for most of the gain?
    1. Show that when the gradient stays constant, SGD with momentum
       $\beta$ at rate $\eta$ eventually takes the same steps as plain SGD
       at rate $\eta / (1 - \beta)$. Do your runs agree?
    1. Repeat the comparison for `ResNet18`. Does it gain as much from
       the change of optimizer?

1. [extended] **Proxy reliability.** ● The method of
   that section assumes that accuracy after a
   few epochs predicts final accuracy and that results at small scale carry
   over to larger networks. Test both with eight configurations sampled from
   a two-stage AnyNet space with $d_i \in \{2, 4, 6\}$,
   $c_i \in \{16, 32, 64, 128\}$ subject to $c_1 \le c_2$, a number of groups
   in $\{4, 8\}$ shared by both stages (the `groups` argument of
   `d2l.ResNeXtBlock`), and $b = 1$. The *rank correlation* of two rankings
   is the correlation between the positions the networks occupy in them.
    1. Train each configuration for 10 epochs on Fashion-MNIST at
       $96 \times 96$, recording validation accuracy after every epoch.
       Compute the rank correlation between the ranking after 1, 2, and 5
       epochs and the ranking after 10 epochs.
    1. Repeat the eight runs with every width halved and the input resized
       to $48 \times 48$. Compute the rank correlation between the final
       rankings at small and at full scale.
    1. Which of the two assumptions holds up better in your experiment? What
       would a failure of the weaker one do to conclusions drawn from short,
       small-scale runs such as those behind the figure?

1. [extended] **A ConvNeXt design space.** ● The design spaces of
   the figure were built from ResNeXt blocks. Build a
   two-stage AnyNet space from the `ConvNeXtBlock` of that section
   instead, with the stem and downsampling layer of `ConvNeXt`. Stage $i$
   has depth $d_i \in \{1, 2, 4\}$, width $c_i \in \{32, 64, 128\}$, and an
   expansion ratio in $\{2, 4\}$. `ConvNeXtBlock` fixes the expansion ratio
   at 4, so add it as an argument.
    1. Which of the AnyNet parameters $d_i$, $c_i$, $g_i$, and $b_i$ have a
       counterpart in this space, and which does the block fix?
    1. Sample 16 networks from the unconstrained space, 16 with one
       expansion ratio shared by both stages, and 16 with $c_1 \le c_2$ and
       $d_1 \le d_2$. Train each for two epochs on Fashion-MNIST at
       $96 \times 96$ and plot the three empirical CDFs the equation.
       Which differences are resolved at 16 samples per space?
    1. Which of the RegNet design principles survive the change of block?

[Discussions](https://d2l.discourse.group/t/7463)